In [1]:
import csv
import json
import sys
from pathlib import Path

import pandas as pd


In [2]:
EXP_DIR = Path.cwd()
REPO_ROOT = EXP_DIR.parents[1]

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from nli_lib.constants import (
    MATERIAL_TOPIC_NA_SENTINEL,
    PAIRING_DIR,
    REPORTS_MAIN_TXT,
    VARIANT_RUNS_DIR,
    VARIANTS,
    DISAGREEMENTS_CSV,
    DISCLOSURE_INDEX_CSV,
    PAIRING_CSV,
    VERDICTS_LONG_CSV,
)

VERDICTS_LONG_OUT   = VERDICTS_LONG_CSV
DISCLOSURE_INDEX_OUT = DISCLOSURE_INDEX_CSV
PAIRING_OUT         = PAIRING_CSV
DISAGREEMENTS_OUT   = DISAGREEMENTS_CSV

LLM_PHASES = ["phase3", "phase5", "phase6"]
REAL_STATUSES = {"pass", "partial", "fail", "no_evidence"}

TAG_OMITTED = "_omitted_"
TAG_EXTERNAL = "_external_ref_"
TAG_NO_PACK = "_no_pack_consistent_"
TAG_GCI_ERROR = "_gci_error_"
TAG_UNRECOVERABLE = "_unrecoverable_"
TAG_MISSING = "_missing_"

PAIRING_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
report_ids = REPORTS_MAIN_TXT.read_text(encoding="utf-8").split()
assert VARIANT_RUNS_DIR.exists(), f"{VARIANT_RUNS_DIR} không tồn tại"
print(f"{len(report_ids)} report_ids; variants={VARIANTS}; phases={LLM_PHASES}")


16 report_ids; variants=['a0', 'a1', 'a2', 'v_new']; phases=['phase3', 'phase5', 'phase6']


## 1. Chuẩn hóa JSON từ 4 variant -> CSV

**Output:**

- `verdicts_long.csv` | Một **requirement** — `status`, rationale, citations, … |
- `disclosure_index.csv` | Một **disclosure** — overall, notes, cờ omitted / no page pack / … |


In [4]:
def load_compliance_report(variant: str, report_id: str) -> dict | None:
    path = VARIANT_RUNS_DIR / variant / report_id / "compliance_report.json"
    if not path.exists():
        return None
    with path.open(encoding="utf-8") as fh:
        return json.load(fh)

def phase_disclosure_verdicts(report: dict, phase_key: str) -> list[dict]:
    phase = report["phase_results"].get(phase_key)
    if not phase:
        return []
    return phase["artifacts"]["disclosure_verdicts"]


In [5]:
VERDICT_COLUMNS = [
    "variant", "report_id", "phase", "phase_disclosure_idx", "occurrence_idx",
    "standard_id", "disclosure_id", "material_topic", "requirement_id",
    "status", "decision_path", "rationale", "citations_json", "confidence", "source",
]

def emit_verdict_rows(variant: str, report_id: str, report: dict) -> list[dict]:
    rows: list[dict] = []
    for phase_key in LLM_PHASES:
        seen: dict[tuple, int] = {}
        for d_idx, disc in enumerate(phase_disclosure_verdicts(report, phase_key)):
            disclosure_id = disc.get("disclosure_id")
            standard_id = disc.get("standard_id")
            material_topic = disc.get("material_topic")
            for rv in disc.get("requirement_verdicts") or []:
                req_id = rv.get("requirement_id")
                grp_key = (disclosure_id, material_topic, req_id)
                occ = seen.get(grp_key, 0)
                seen[grp_key] = occ + 1
                rows.append({
                    "variant": variant, "report_id": report_id, "phase": phase_key,
                    "phase_disclosure_idx": d_idx, "occurrence_idx": occ,
                    "standard_id": standard_id, "disclosure_id": disclosure_id,
                    "material_topic": material_topic, "requirement_id": req_id,
                    "status": rv.get("status"), "decision_path": rv.get("decision_path"),
                    "rationale": rv.get("rationale") or "",
                    "citations_json": json.dumps(rv.get("citations") or [], ensure_ascii=False),
                    "confidence": rv.get("confidence"), "source": rv.get("source"),
                })
    return rows


In [6]:
DISCLOSURE_COLUMNS = [
    "variant", "report_id", "phase", "phase_disclosure_idx", "occurrence_idx",
    "standard_id", "disclosure_id", "material_topic", "overall", "notes",
    "evidence_unrecoverable", "n_requirement_verdicts",
    "is_omitted", "is_external_ref", "is_no_page_pack", "is_empty_verdicts",
]

def classify_disclosure(disc: dict) -> dict:
    overall = (disc.get("overall") or "").lower()
    notes = (disc.get("notes") or "").lower()
    n_rv = len(disc.get("requirement_verdicts") or [])
    return {
        "overall": disc.get("overall"), "notes": disc.get("notes") or "",
        "evidence_unrecoverable": bool(disc.get("evidence_unrecoverable")),
        "n_requirement_verdicts": n_rv,
        "is_omitted": overall == "omitted" or "omitted" in notes,
        "is_external_ref": overall == "external_verification",
        "is_no_page_pack": "no page pack" in notes,
        "is_empty_verdicts": n_rv == 0,
    }

def emit_disclosure_rows(variant: str, report_id: str, report: dict) -> list[dict]:
    rows: list[dict] = []
    for phase_key in LLM_PHASES:
        seen: dict[tuple, int] = {}
        for d_idx, disc in enumerate(phase_disclosure_verdicts(report, phase_key)):
            disclosure_id = disc.get("disclosure_id")
            material_topic = disc.get("material_topic")
            grp_key = (disclosure_id, material_topic)
            occ = seen.get(grp_key, 0)
            seen[grp_key] = occ + 1
            rows.append({
                "variant": variant, "report_id": report_id, "phase": phase_key,
                "phase_disclosure_idx": d_idx, "occurrence_idx": occ,
                "standard_id": disc.get("standard_id"),
                "disclosure_id": disclosure_id, "material_topic": material_topic,
                **classify_disclosure(disc),
            })
    return rows


## 2. Driver: lặp variant × report → flatten

Đếm verdict + disclosure rows mỗi variant, log report nào thiếu artifact (`compliance_report.json` chưa chạy hết).


In [7]:
verdict_rows: list[dict] = []
disclosure_rows: list[dict] = []
n_loaded = 0
n_missing = 0
per_variant: dict[str, dict[str, int]] = {v: {"verdicts": 0, "disclosures": 0, "missing": 0} for v in VARIANTS}

for variant in VARIANTS:
    for rid in report_ids:
        report = load_compliance_report(variant, rid)
        if report is None:
            n_missing += 1
            per_variant[variant]["missing"] += 1
            print(f"  [warn] {variant}/{rid}: compliance_report.json missing")
            continue
        n_loaded += 1
        v_rows = emit_verdict_rows(variant, rid, report)
        d_rows = emit_disclosure_rows(variant, rid, report)
        verdict_rows.extend(v_rows)
        disclosure_rows.extend(d_rows)
        per_variant[variant]["verdicts"] += len(v_rows)
        per_variant[variant]["disclosures"] += len(d_rows)

print()
for v, c in per_variant.items():
    print(f"  {v:8s}  verdicts={c['verdicts']:5d}  disclosures={c['disclosures']:4d}  missing={c['missing']}")
print(f"\nLoaded {n_loaded} reports (missing: {n_missing})")



  a0        verdicts= 8670  disclosures=1845  missing=0
  a1        verdicts= 8660  disclosures=1845  missing=0
  a2        verdicts= 8670  disclosures=1845  missing=0
  v_new     verdicts= 8670  disclosures=1845  missing=0

Loaded 64 reports (missing: 0)


## 3. Lưu long-format CSV


In [8]:
with VERDICTS_LONG_OUT.open("w", encoding="utf-8", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=VERDICT_COLUMNS, lineterminator="\n")
    writer.writeheader()
    for row in verdict_rows:
        writer.writerow(row)

with DISCLOSURE_INDEX_OUT.open("w", encoding="utf-8", newline="") as fh:
    writer = csv.DictWriter(fh, fieldnames=DISCLOSURE_COLUMNS, lineterminator="\n")
    writer.writeheader()
    for row in disclosure_rows:
        writer.writerow(row)

print(f"Wrote {len(verdict_rows):,} rows -> {VERDICTS_LONG_OUT.name}")
print(f"Wrote {len(disclosure_rows):,} rows -> {DISCLOSURE_INDEX_OUT.name}")


Wrote 34,670 rows -> verdicts_long.csv
Wrote 7,380 rows -> disclosure_index.csv


## 4. Build pairing: long → wide

Đọc lại 2 long-format CSV (roundtrip qua disk để chứng minh tính độc lập của 2 stage). Normalize NaN ở boundary: `material_topic` nullable trong JSON → sentinel `__none__`; `notes` luôn string trong JSON nhưng `""` → NaN sau roundtrip CSV → bù lại "" cho downstream truy cập trực tiếp không cần `.get()`.


In [9]:
verdicts = pd.read_csv(VERDICTS_LONG_OUT)
disclosures = pd.read_csv(DISCLOSURE_INDEX_OUT)
for df in (verdicts, disclosures):
    df["material_topic"] = df["material_topic"].fillna(MATERIAL_TOPIC_NA_SENTINEL)
disclosures["notes"] = disclosures["notes"].fillna("")
print(f"verdicts: {len(verdicts):,} rows; disclosures: {len(disclosures):,} rows")


verdicts: 34,670 rows; disclosures: 7,380 rows


## 5. Build universe + pivot 4 variant column

Universe = tập unique `(report, phase, disclosure, material_topic, requirement, occurrence)` xuất hiện ở **ít nhất 1** variant. Mỗi variant merge vào universe theo left-join → cell trống = variant không emit verdict cho key đó (sẽ resolve sau).


In [10]:
PER_VARIANT_FIELDS = ["status", "decision_path", "rationale", "citations_json", "confidence", "source"]
JOIN_KEYS = ["report_id", "phase", "disclosure_id", "material_topic", "requirement_id", "occurrence_idx"]
DISC_KEY_COLS = ["report_id", "disclosure_id", "material_topic", "occurrence_idx"]

universe = (
    verdicts[JOIN_KEYS + ["standard_id"]]
    .drop_duplicates(subset=JOIN_KEYS)
    .reset_index(drop=True)
)
print(f"universe: {len(universe):,} unique req-keys")

pivot = universe.copy()
for v in VARIANTS:
    sub = verdicts.loc[verdicts["variant"] == v, JOIN_KEYS + PER_VARIANT_FIELDS]
    sub = sub.rename(columns={f: f"{v}_{f}" for f in PER_VARIANT_FIELDS})
    sub = sub.drop_duplicates(subset=JOIN_KEYS, keep="last")
    pivot = pivot.merge(sub, on=JOIN_KEYS, how="left")
print(f"pivot: {len(pivot):,} rows × {len(pivot.columns)} cols")


universe: 8,670 unique req-keys
pivot: 8,670 rows × 31 cols


## 6. Resolve status tags (omitted / external / unrecoverable / ...)

Sau merge, ô `{v}_status` có thể là:
- **str** (1 trong REAL_STATUSES) → giữ nguyên.
- **NaN** (variant không emit verdict) → tra `disclosure_index` để gán tag chính xác:
  - `_omitted_` — disclosure overall=omitted hoặc notes chứa "omitted"
  - `_external_ref_` — overall=external_verification
  - `_gci_error_` — notes chứa "GCI row has missing"
  - `_no_pack_consistent_` — không có page pack VÀ tất cả variant đều empty (data gap)
  - `_unrecoverable_` — không có page pack hoặc empty verdicts mà variant khác lại có
  - `_missing_` — fallback


In [11]:
disc_lookup = disclosures.set_index(["variant"] + DISC_KEY_COLS)[[
    "is_empty_verdicts", "is_omitted", "is_external_ref", "is_no_page_pack", "notes", "overall",
]].to_dict(orient="index")

g = disclosures.groupby(DISC_KEY_COLS)["is_empty_verdicts"]
disc_consistency = {key: {"all_empty": bool(v)} for key, v in (g.sum() == g.count()).items()}

def resolve_status_for_variant(raw_status, variant, report_id, disclosure_id, material_topic, occurrence_idx):
    if isinstance(raw_status, str):
        return raw_status
    disc_key = (report_id, disclosure_id, material_topic, occurrence_idx)
    cons = disc_consistency.get(disc_key)
    info = disc_lookup.get((variant, *disc_key))
    if cons is None or info is None:
        return TAG_MISSING
    if info["is_omitted"]:
        return TAG_OMITTED
    if info["is_external_ref"]:
        return TAG_EXTERNAL
    if "GCI row has missing" in info["notes"]:
        return TAG_GCI_ERROR
    if info["is_no_page_pack"]:
        return TAG_NO_PACK if cons["all_empty"] else TAG_UNRECOVERABLE
    if info["is_empty_verdicts"]:
        return TAG_UNRECOVERABLE
    return TAG_MISSING

keys = ["report_id", "disclosure_id", "material_topic", "occurrence_idx"]
for v in VARIANTS:
    pivot[f"{v}_status_resolved"] = [
        resolve_status_for_variant(raw, v, rid, did, mt, int(occ))
        for raw, rid, did, mt, occ in zip(pivot[f"{v}_status"], *(pivot[k] for k in keys))
    ]


## 7. Compute disagreement flags + lưu pairing.csv + disagreements.csv

- `any_disagree_primary` = ≥2 distinct **REAL** statuses (không tính tag) trong 4 variant.
- `any_unrecoverable_primary` = có ít nhất 1 variant `_unrecoverable_`.
- `disagreements.csv` = subset của pairing với disagree=True AND unrecoverable=False (case có ý nghĩa cho arbiter).


In [ ]:
primary_cols = [f"{v}_status_resolved" for v in VARIANTS]
primary = pivot[primary_cols]

real_only = primary.where(primary.isin(REAL_STATUSES))
pivot["any_disagree_primary"] = real_only.nunique(axis=1) > 1
pivot["any_unrecoverable_primary"] = (primary == TAG_UNRECOVERABLE).any(axis=1)
pivot["any_gci_error"] = (primary == TAG_GCI_ERROR).any(axis=1)
pivot["disagreement_pattern"] = primary.astype(str).agg("|".join, axis=1)

pivot.to_csv(PAIRING_OUT, index=False)
print(f"Wrote {len(pivot):,} rows -> {PAIRING_OUT.name}")

dis_mask = pivot["any_disagree_primary"] & ~pivot["any_unrecoverable_primary"]
pivot.loc[dis_mask].to_csv(DISAGREEMENTS_OUT, index=False)
print(f"Wrote {int(dis_mask.sum()):,} rows -> {DISAGREEMENTS_OUT.name}")

print()
print(f"Total:           {len(pivot):,}")
print(f"Disagreements:  {int(pivot['any_disagree_primary'].sum()):,}  ({100 * pivot['any_disagree_primary'].mean():.1f}%)")


Wrote 8,670 rows -> pairing.csv
Wrote 2,361 rows -> disagreements.csv

=== Summary ===
Total req-keys:           8,670
Disagreements (primary):  2,365  (27.3%)
Unrecoverable (primary):  10  (0.12%)
GCI data errors:          0  (0.00%)
